# Joshi Part 7: PayOff Classes and Greeks (Rust)

Rust kernel version of `07_joshi_payoffs_greeks.ipynb`.

## Setup

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }
:dep time = { version = "0.3", features = ["macros"] }

In [ ]:
use time::macros::date;
use RustQuant::instruments::options::*;
use RustQuant::instruments::*;
use RustQuant::stochastics::*;

let spot = 100.0;
let strike = 100.0;
let rate = 0.05;
let vol = 0.20;
let expiry = date!(2027 - 03 - 22);
let n_paths = 100_000;

let gbm = GeometricBrownianMotion::new(rate, vol);
let config = StochasticProcessConfig::new(
    spot, 0.0, 1.0, 252, StochasticScheme::EulerMaruyama, n_paths, true, None,
);
let contract = OptionContractBuilder::default()
    .type_flag(TypeFlag::Call)
    .exercise_flag(ExerciseFlag::European { expiry })
    .strike_flag(Some(StrikeFlag::Fixed))
    .build().unwrap();

## 1. Vanilla Call and Put

In [ ]:
let call = EuropeanVanillaOption::new(strike, expiry, TypeFlag::Call);
let put = EuropeanVanillaOption::new(strike, expiry, TypeFlag::Put);

let call_price = call.price_monte_carlo(&gbm, &config, rate);
let put_price = put.price_monte_carlo(&gbm, &config, rate);

println!("Vanilla Call: {:.4}", call_price);
println!("Vanilla Put:  {:.4}", put_price);
println!("C-P = {:.4}, S-K*exp(-rT) = {:.4}",
    call_price - put_price,
    spot - strike * (-rate * 1.0_f64).exp());

## 2. Digital (Binary) Options

In [ ]:
let cash_call = BinaryOption::new(contract.clone(), BinaryType::CashOrNothing, strike);
let asset_call = BinaryOption::new(contract.clone(), BinaryType::AssetOrNothing, strike);

println!("Cash-or-Nothing Call:  {:.4}", cash_call.price_monte_carlo(&gbm, &config, rate));
println!("Asset-or-Nothing Call: {:.4}", asset_call.price_monte_carlo(&gbm, &config, rate));

## 3. Double Digital (Supershare)

In [ ]:
let ss = SupershareOption::new(90.0, 110.0);
println!("Supershare (90, 110): {:.4}", ss.price_monte_carlo(&gbm, &config, rate));

## 4. Power and Log Options

In [ ]:
let power = PowerOption::new(contract.clone(), strike, 2.0);
let log_opt = LogOption::new(strike);

println!("Power Option (n=2): {:.4}", power.price_monte_carlo(&gbm, &config, rate));
println!("Log Option:         {:.4}", log_opt.price_monte_carlo(&gbm, &config, rate));

## 5. Analytic Greeks (BSM)

In [ ]:
let bsm = BlackScholesMertonBuilder::default()
    .underlying_price(spot).strike_price(strike).volatility(vol)
    .risk_free_rate(rate).cost_of_carry(rate)
    .expiration_date(expiry).option_type(TypeFlag::Call)
    .build().unwrap();

println!("BSM Call Greeks:");
println!("  Price  = {:.6}", bsm.price());
println!("  Delta  = {:.6}", bsm.delta());
println!("  Gamma  = {:.6}", bsm.gamma());
println!("  Vega   = {:.6}", bsm.vega());
println!("  Theta  = {:.6}", bsm.theta());
println!("  Rho    = {:.6}", bsm.rho());
println!("  Vanna  = {:.6}", bsm.vanna());
println!("  Vomma  = {:.6}", bsm.vomma());
println!("  Charm  = {:.6}", bsm.charm());
println!("  Zomma  = {:.6}", bsm.zomma());
println!("  Speed  = {:.6}", bsm.speed());
println!("  Colour = {:.6}", bsm.colour());
println!("  Lambda = {:.6}", bsm.lambda());

## 6. Strike Sensitivity

In [ ]:
println!("{:<8} {:<12} {:<12} {:<12}", "Strike", "Vanilla", "Digital", "Supershare");
println!("{}", "-".repeat(44));
for k in (80..=120).step_by(10) {
    let k = k as f64;
    let v = EuropeanVanillaOption::new(k, expiry, TypeFlag::Call);
    let c = OptionContractBuilder::default()
        .type_flag(TypeFlag::Call).exercise_flag(ExerciseFlag::European { expiry })
        .strike_flag(Some(StrikeFlag::Fixed)).build().unwrap();
    let d = BinaryOption::new(c, BinaryType::CashOrNothing, k);
    let ss = SupershareOption::new(k - 5.0, k + 5.0);
    println!("{:<8.0} {:<12.4} {:<12.4} {:<12.4}",
        k,
        v.price_monte_carlo(&gbm, &config, rate),
        d.price_monte_carlo(&gbm, &config, rate),
        ss.price_monte_carlo(&gbm, &config, rate));
}